# Join, Reshape, and Aggregate

**Project question:** How can I construct exactly one defensible row per store-week?

By the end of this notebook, you should be able to:

- apply documented cleaning rules and record row-count changes
- enforce many-to-one join cardinality and inspect unmatched keys
- verify the final observational unit after aggregation

This notebook is a demonstration, not a homework assignment. The data are
synthetic and were generated for teaching; numerical results should not be
interpreted as evidence about a real organization or population.

In [1]:

from lite_setup import ensure_packages
await ensure_packages()

Using the current Python environment.


In [2]:

import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path("data")
if not DATA.exists():
    raise FileNotFoundError("Open this notebook from the JupyterLite files root so data/ is available.")

In [3]:
sales = pd.read_csv(DATA / 'retail_sales_messy.csv')
stores = pd.read_csv(DATA / 'store_metadata.csv')
weather = pd.read_csv(DATA / 'weather_weekly.csv')

We make the cleaning policy explicit: remove an exact duplicate, convert the cents entry to dollars, and exclude rows with missing required values, nonpositive price, invalid date, or the deliberately implausible units outlier.

In [4]:
row_counts = [{'stage': 'raw', 'rows': len(sales)}]
sales_clean = sales.drop_duplicates().copy()
row_counts.append({'stage': 'after exact-duplicate removal', 'rows': len(sales_clean)})
for col in ['units', 'revenue', 'price', 'week', 'promotion']:
    sales_clean[col] = pd.to_numeric(sales_clean[col], errors='coerce')
sales_clean['date'] = pd.to_datetime(sales_clean['date'], errors='coerce')
sales_clean.loc[sales_clean['price'] > 50, 'price'] /= 100
invalid = (
    sales_clean[['units', 'revenue', 'price', 'week']].isna().any(axis=1)
    | sales_clean['date'].isna()
    | sales_clean['price'].le(0)
    | sales_clean['units'].gt(1000)
)
sales_clean = sales_clean.loc[~invalid].copy()
row_counts.append({'stage': 'after documented validity rules', 'rows': len(sales_clean)})
pd.DataFrame(row_counts)

,stage,rows
0,raw,98
1,after exact-duplicate removal,97
2,after documented validity rules,93


Before joining, verify that each lookup key is unique. `validate='many_to_one'` makes pandas stop rather than silently multiplying rows if a lookup table has duplicate keys.

In [5]:
assert not stores['store_id'].duplicated().any(), 'store_metadata must have one row per store_id'
assert not weather.duplicated(['region', 'week']).any(), 'weather must have one row per region-week'

joined = sales_clean.merge(
    stores, on='store_id', how='left', indicator='store_join', validate='many_to_one'
)
store_join_audit = joined['store_join'].value_counts().rename_axis('status').to_frame('rows')
unused_store_keys = sorted(set(stores['store_id']) - set(sales_clean['store_id']))
store_join_audit, unused_store_keys

(            rows
 status          
 both          92
 left_only      1
 right_only     0,
 ['S999'])

In [6]:
joined = joined.loc[joined['store_join'].eq('both')].drop(columns='store_join')
joined = joined.merge(
    weather, on=['region', 'week'], how='left', indicator='weather_join', validate='many_to_one'
)
weather_join_audit = joined['weather_join'].value_counts().rename_axis('status').to_frame('rows')
weather_join_audit

,rows
status,
both,92
left_only,0
right_only,0


The unmatched sales store is excluded because store attributes are required. The unused `S999` metadata row is harmless but documented. No weather rows should be unmatched.

In [7]:
joined = joined.loc[joined['weather_join'].eq('both')].drop(columns='weather_join')
model_table = joined.groupby(['store_id', 'week'], as_index=False).agg(
    week_start=('date', 'min'),
    units=('units', 'sum'),
    revenue=('revenue', 'sum'),
    avg_price=('price', 'mean'),
    promotion=('promotion', 'max'),
    store_size_sqft=('store_size_sqft', 'first'),
    temperature=('temperature', 'mean'),
    precipitation=('precipitation', 'mean'),
    region=('region', 'first'),
)
assert not model_table.duplicated(['store_id', 'week']).any()
print(f'Final store-week rows: {len(model_table)}')
model_table.head()

Final store-week rows: 92


,store_id,week,week_start,units,revenue,avg_price,promotion,store_size_sqft,temperature,precipitation,region
0,S001,1,2026-01-01,166.0,1641.74,9.89,0,1706.0,47.5,0.28,North
1,S001,2,2026-01-08,167.0,1573.14,9.42,0,1706.0,49.2,0.59,North
2,S001,3,2026-01-15,177.0,1612.47,9.11,0,1706.0,48.9,0.48,North
3,S001,4,2026-01-22,161.0,1503.74,9.34,0,1706.0,50.6,0.04,North
4,S001,5,2026-02-01,225.0,2074.50,9.22,1,1706.0,53.1,0.99,North


**Interpretation:** The assertion verifies the claimed observational unit. Row-count and join-audit tables belong in the project record because a successful merge alone does not prove the keys or exclusions were appropriate.